In [ ]:
# --- Path setup for local library imports ---
import os
import sys

# Get current working directory (where the notebook is)
current_dir = os.getcwd()

# Go one level up (where 'my_ml_lib' folder is located)
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))

# Add parent directory to Python path if not already added
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
 


Project root added to sys.path: d:\FOML\Final_BoilerPlate


In [7]:
from my_ml_lib.datasets._loaders import load_spambase
from my_ml_lib.preprocessing import StandardScaler
from my_ml_lib.linear_models.classification import LogisticRegression
from my_ml_lib.model_selection._split import train_test_split


In [11]:
def k_fold_indices(n_samples, n_splits=5, random_state=None):
    if random_state is not None:
        np.random.seed(random_state)
    indices = np.arange(n_samples)
    np.random.shuffle(indices)
    fold_sizes = np.full(n_splits, n_samples // n_splits, dtype=int)
    fold_sizes[: n_samples % n_splits] += 1
    current = 0
    folds = []
    for fold_size in fold_sizes:
        start, stop = current, current + fold_size
        folds.append(indices[start:stop])
        current = stop
    return folds


def cross_val_score(model_class, X, y, alpha_values, k=5, random_state=42):
    """
    Perform K-Fold cross-validation for different alpha values.
    Returns dict mapping alpha -> mean accuracy.
    """
    n_samples = X.shape[0]
    folds = k_fold_indices(n_samples, n_splits=k, random_state=random_state)
    results = {}

    for alpha in alpha_values:
        accuracies = []
        for i in range(k):
            val_idx = folds[i]
            train_idx = np.hstack([folds[j] for j in range(k) if j != i])
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            model = model_class(alpha=alpha, max_iter=100)
            model.fit(X_train, y_train)
            acc = np.mean(model.predict(X_val) == y_val)
            accuracies.append(acc)

        results[alpha] = np.mean(accuracies)
    return results


# =============================================================
# 1. Load Data
# =============================================================
X, y = load_spambase("../data/spambase.data")
print(f"Loaded Spambase dataset: X={X.shape}, y={y.shape}")

# =============================================================
# 2. Split into Train/Test
# =============================================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# =============================================================
# 3. Define alpha values
# =============================================================
alpha_values = [0.01, 0.1, 1, 10, 100]

# =============================================================
# 4. Train Logistic Regression on RAW data
# =============================================================
print("\n--- Raw Data (no scaling) ---")
raw_cv_results = cross_val_score(LogisticRegression, X_train, y_train, alpha_values)
best_alpha_raw = max(raw_cv_results, key=raw_cv_results.get)
print("Cross-Validation Accuracies:", raw_cv_results)
print(f"Best alpha (raw): {best_alpha_raw}")

# Train final model on full training set
model_raw = LogisticRegression(alpha=best_alpha_raw, max_iter=100)
model_raw.fit(X_train, y_train)
train_acc_raw = np.mean(model_raw.predict(X_train) == y_train)
test_acc_raw = np.mean(model_raw.predict(X_test) == y_test)
print(f"Train Accuracy (raw): {train_acc_raw:.4f}")
print(f"Test Accuracy  (raw): {test_acc_raw:.4f}")

# =============================================================
# 5. Train Logistic Regression on STANDARDIZED data
# =============================================================
print("\n--- Standardized Data ---")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

scaled_cv_results = cross_val_score(LogisticRegression, X_train_scaled, y_train, alpha_values)
best_alpha_scaled = max(scaled_cv_results, key=scaled_cv_results.get)
print("Cross-Validation Accuracies:", scaled_cv_results)
print(f"Best alpha (scaled): {best_alpha_scaled}")

# Train final model
model_scaled = LogisticRegression(alpha=best_alpha_scaled, max_iter=100)
model_scaled.fit(X_train_scaled, y_train)
train_acc_scaled = np.mean(model_scaled.predict(X_train_scaled) == y_train)
test_acc_scaled = np.mean(model_scaled.predict(X_test_scaled) == y_test)
print(f"Train Accuracy (scaled): {train_acc_scaled:.4f}")
print(f"Test Accuracy  (scaled): {test_acc_scaled:.4f}")

# =============================================================
# 6. Summary Table
# =============================================================
print("\n===== Summary =====")
print(f"Raw Data    : best α={best_alpha_raw}, Train Acc={train_acc_raw:.3f}, Test Acc={test_acc_raw:.3f}")
print(f"Standardized: best α={best_alpha_scaled}, Train Acc={train_acc_scaled:.3f}, Test Acc={test_acc_scaled:.3f}")


Loaded Spambase dataset: X=(4601, 57), y=(4601,)

--- Raw Data (no scaling) ---


d:\FOML\Final_BoilerPlate\my_ml_lib\linear_models\classification\_logistic.py:27: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-z))


Cross-Validation Accuracies: {0.01: np.float64(0.9225742581558609), 0.1: np.float64(0.9255622824612117), 1: np.float64(0.9231192112559732), 10: np.float64(0.9195880773995635), 100: np.float64(0.885358902129668)}
Best alpha (raw): 0.1
Train Accuracy (raw): 0.9326
Test Accuracy  (raw): 0.9207

--- Standardized Data ---
Cross-Validation Accuracies: {0.01: np.float64(0.9225738894460503), 0.1: np.float64(0.9228471034157277), 1: np.float64(0.9214884077635539), 10: np.float64(0.9193152321396967), 100: np.float64(0.9060018582974456)}
Best alpha (scaled): 0.1
Train Accuracy (scaled): 0.9326
Test Accuracy  (scaled): 0.9196

===== Summary =====
Raw Data    : best α=0.1, Train Acc=0.933, Test Acc=0.921
Standardized: best α=0.1, Train Acc=0.933, Test Acc=0.920
